# 03 — Native Precipitation QC (FIXED)

**No training raster is resampled here.**  
This is deliberate: training/validation station values must be extracted from each
precipitation product at its **native grid**, following the base-paper logic.

The notebook validates monthly coverage, units/value plausibility, CRS and NoData.

In [1]:
from pathlib import Path
import warnings

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "data").exists():
            return candidate
    raise FileNotFoundError(
        "Project root not found. Run this notebook from inside the repository."
    )

PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
MODEL_DIR = PROJECT_ROOT / "models"

for d in [INTERIM_DIR, PROCESSED_DIR, OUTPUT_DIR, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT =", PROJECT_ROOT)

PROJECT_ROOT = E:\Geospatial\Precipitation-Downscaling-Khulna\Machine-Learning-Based-Precipitation-Downscaling-over-Khulna-District-Bangladesh


In [3]:
import re
import numpy as np
import pandas as pd
import rasterio
from pathlib import Path

# ============================================================
# PRECIPITATION DATASET -> ACTUAL FOLDER NAME
# ============================================================

PRECIP_FOLDERS = {
    "CCS": "CCS",
    "PDIR": "PDIR",
    "GSMaP_MVK": "GSMaP_MVK",
    "CDR": "CDR",                              # PERSIANN-CDR
    "CHIRPS": "CHIRPS_TIFF_2017_2022",
    "IMERG": "IMERG_Monthly",
    "GSMaP_Gauge": "GSMaP_Gauge_v7",
    "ERA5": "ERA5_TIFF",
}

YEARS = range(2017, 2023)


# ============================================================
# PARSE YEAR + MONTH FROM FILE NAME
# ============================================================

def parse_ym(name):
    stem = Path(name).stem

    patterns = [
        r"(?<!\d)(20\d{2})[_-](0?[1-9]|1[0-2])(?!\d)",
        r"(?<!\d)(20\d{2})(0[1-9]|1[0-2])(?!\d)"
    ]

    for pat in patterns:
        m = re.search(pat, stem)

        if m:
            return int(m.group(1)), int(m.group(2))

    return None


# ============================================================
# LIST MONTHLY RASTERS
# ============================================================

def list_monthly(folder):

    files = sorted([
        *folder.rglob("*.tif"),
        *folder.rglob("*.tiff")
    ])

    out = {}

    for p in files:

        ym = parse_ym(p.name)

        if ym:

            if ym in out:
                raise ValueError(
                    f"Duplicate raster for {folder.name} {ym}:\n"
                    f"{out[ym]}\n{p}"
                )

            out[ym] = p

    return out


# ============================================================
# CHECK ALL PRECIPITATION PRODUCTS
# ============================================================

precip_root = RAW_DIR / "precipitation"

maps = {}

for product, folder_name in PRECIP_FOLDERS.items():

    folder = precip_root / folder_name

    if not folder.exists():
        raise FileNotFoundError(
            f"Required precipitation folder missing:\n{folder}"
        )

    maps[product] = list_monthly(folder)

    missing = [
        (y, m)
        for y in YEARS
        for m in range(1, 13)
        if (y, m) not in maps[product]
    ]

    print(
        f"{product:15s}: "
        f"{len(maps[product]):2d} parsed monthly rasters; "
        f"missing={len(missing)}"
    )

    if missing:
        print("   Missing:", missing)


print("\n==========================================")
print("PRECIPITATION MONTHLY COVERAGE CHECK DONE")
print("==========================================")

CCS            : 72 parsed monthly rasters; missing=0
PDIR           : 72 parsed monthly rasters; missing=0
GSMaP_MVK      : 72 parsed monthly rasters; missing=0
CDR            : 72 parsed monthly rasters; missing=0
CHIRPS         : 72 parsed monthly rasters; missing=0
IMERG          : 72 parsed monthly rasters; missing=0
GSMaP_Gauge    : 72 parsed monthly rasters; missing=0
ERA5           : 72 parsed monthly rasters; missing=0

PRECIPITATION MONTHLY COVERAGE CHECK DONE


In [4]:
rows = []
for product, monthly in maps.items():
    for (y,m), p in sorted(monthly.items()):
        with rasterio.open(p) as src:
            a = src.read(1, masked=True)
            v = a.compressed().astype("float64")
            neg = int(np.sum(v < 0)) if v.size else 0
            rows.append({
                "product":product, "year":y, "month":m, "path":str(p),
                "crs":str(src.crs), "width":src.width, "height":src.height,
                "res_x":src.res[0], "res_y":src.res[1],
                "nodata":src.nodata,
                "valid_pct":100*v.size/a.size if a.size else np.nan,
                "min":float(np.nanmin(v)) if v.size else np.nan,
                "max":float(np.nanmax(v)) if v.size else np.nan,
                "mean":float(np.nanmean(v)) if v.size else np.nan,
                "negative_valid_pixels":neg,
                "bounds":str(tuple(src.bounds)),
            })

qc = pd.DataFrame(rows)
display(qc.head(20))
qc.to_csv(PROCESSED_DIR / "precipitation_native_qc.csv", index=False)

# Negative valid precipitation is suspicious; negative NoData values masked by rasterio are not counted.
bad_neg = qc[qc["negative_valid_pixels"] > 0]
if len(bad_neg):
    print("WARNING: Negative valid precipitation values detected. Confirm units/NoData metadata.")
    display(bad_neg[["product","year","month","min","negative_valid_pixels","path"]])

# Monthly values above this are not auto-deleted, only flagged for manual unit review.
very_high = qc[qc["max"] > 3000]
if len(very_high):
    print("WARNING: Some monthly raster maxima exceed 3000 mm. Verify source unit/aggregation.")
    display(very_high[["product","year","month","max","path"]])

print("\nIMPORTANT: confirm that every source raster already represents monthly precipitation in mm/month.")
print("This notebook intentionally performs NO resampling of training/validation rasters.")

,product,year,month,path,crs,width,height,res_x,res_y,nodata,valid_pct,min,max,mean,negative_valid_pixels,bounds
0,CCS,2017,1,E:\Geospatial\Precipitation-Downscaling-Khulna...,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,45.37037,0.0,3.0,0.391837,0,"(89.2, 21.639999999999997, 89.8, 23.08)"
1,CCS,2017,2,E:\Geospatial\Precipitation-Downscaling-Khulna...,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,45.37037,0.0,0.0,0.000000,0,"(89.2, 21.639999999999997, 89.8, 23.08)"
2,CCS,2017,3,E:\Geospatial\Precipitation-Downscaling-Khulna...,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,45.37037,0.0,26.0,4.253061,0,"(89.2, 21.639999999999997, 89.8, 23.08)"
3,CCS,2017,4,E:\Geospatial\Precipitation-Downscaling-Khulna...,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,45.37037,18.0,113.0,42.048980,0,"(89.2, 21.639999999999997, 89.8, 23.08)"
4,CCS,2017,5,E:\Geospatial\Precipitation-Downscaling-Khulna...,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,45.37037,15.0,112.0,52.669388,0,"(89.2, 21.639999999999997, 89.8, 23.08)"
5,CCS,2017,6,E:\Geospatial\Precipitation-Downscaling-Khulna...,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,45.37037,454.0,832.0,582.877551,0,"(89.2, 21.639999999999997, 89.8, 23.08)"
6,CCS,2017,7,E:\Geospatial\Precipitation-Downscaling-Khulna...,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,45.37037,410.0,920.0,560.975510,0,"(89.2, 21.639999999999997, 89.8, 23.08)"
7,CCS,2017,8,E:\Geospatial\Precipitation-Downscaling-Khulna...,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,45.37037,210.0,473.0,323.755102,0,"(89.2, 21.639999999999997, 89.8, 23.08)"
8,CCS,2017,9,E:\Geospatial\Precipitation-Downscaling-Khulna...,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,45.37037,226.0,467.0,338.285714,0,"(89.2, 21.639999999999997, 89.8, 23.08)"
9,CCS,2017,10,E:\Geospatial\Precipitation-Downscaling-Khulna...,"GEOGCS[""WGS 84"",DATUM[""World Geodetic System 1...",15,36,0.04,0.04,-99.0,45.37037,135.0,532.0,276.510204,0,"(89.2, 21.639999999999997, 89.8, 23.08)"



IMPORTANT: confirm that every source raster already represents monthly precipitation in mm/month.
This notebook intentionally performs NO resampling of training/validation rasters.


In [5]:
# Paper-style feature set excludes a standalone extra PERSIANN folder.
extra = precip_root / "PERSIANN"
if extra.exists():
    print("NOTE: data/raw/precipitation/PERSIANN exists, but it will NOT be used in Comb1/Comb2.")
    print("CDR is treated as the PERSIANN-CDR feature.")

NOTE: data/raw/precipitation/PERSIANN exists, but it will NOT be used in Comb1/Comb2.
CDR is treated as the PERSIANN-CDR feature.
